In [1]:
from winnow_fcns import *
from sim_utils import *
import numpy as np
from pathlib import Path
import os
import csv

In [2]:
#------------------files---------------------------------------------
current_file = Path.cwd()

init_file = current_file.parent / "initial_cyrus_bits.csv"
ch1_file = current_file.parent / "ch1_cyrus.csv"
ch2_file = current_file.parent / "ch2_cyrus.csv"



def load_data(init_file, ch1_file, ch2_file):
    init = np.loadtxt(init_file, delimiter=',', dtype=int)
    ch1 = np.loadtxt(ch1_file, delimiter=',')
    ch2 = np.loadtxt(ch2_file, delimiter=',')
    return init, ch1, ch2


init_key, ch1, ch2 = load_data(init_file, ch1_file, ch2_file)


In [ ]:



def get_curr_qber(alice_winnow, bob_winnow):
    matches = (np.array(alice_winnow._key_string.bits)  != np.array(bob_winnow._key_string.bits) )

    # Calculate percentage
    percentage = matches.mean() * 100

    return percentage


def winnow_pipeline(block_schedule):
    qber_list = []
    num_pass = []

    alice_key = MockBitBuffer(list(ch1))
    bob_key   = MockBitBuffer(list(ch2))

    #  initialize Alice and Bob
    alice_winnow = Winnow(raw_key=alice_key, perm_seed=42)
    bob_winnow = Winnow(raw_key=bob_key, perm_seed=42)
    #use same seed so they dont shuffle keys differently!!!!

    alice_winnow.set_block_size_schedule(block_schedule)
    bob_winnow.set_block_size_schedule(block_schedule)


    qber = get_curr_qber(alice_winnow, bob_winnow)
    print(f"Qber before EC {qber}%")
    qber_list.append(qber)
    num_pass.append(alice_winnow._pass_number)

    # start first pass
    alice_winnow.first_pass()
    bob_winnow.first_pass()

    success_sample = 0
    total_sample = 0
    with open('corrected/cyrus_test.csv', 'w', newline='') as f:
        # alice calculates syndrome for the first block 
        for i in range(alice_winnow._num_of_blocks):
            block_size = alice_winnow._block_size
            startb = i*block_size
            endb = block_size*(i + 1) 
            alice_syndrome = alice_winnow.get_syndrome(i)

            bob_winnow.fix_with_syndrome(i, alice_syndrome)

            
    qber = get_curr_qber(alice_winnow, bob_winnow)
    print(f"Qber after first pass {qber}%")
    qber_list.append(qber)
    num_pass.append(alice_winnow._pass_number)
   

    while alice_winnow.get_num_remaining_passes() > 0:
        # print("\nAlice internal:", alice_winnow._key_string.bits[:20])
        # print("Bob   internal:", bob_winnow._key_string.bits[:20])


        print(f"Alice seed: {alice_winnow._key_string.seed}")
        print(f"Bob seed:   {bob_winnow._key_string.seed}") 
        block_size  = alice_winnow._block_size
        num_blocks  = alice_winnow._num_of_blocks
        alice_winnow._key_string.discard_parity_bits(block_size, num_blocks)
        bob_winnow._key_string.discard_parity_bits(block_size, num_blocks)
        alice_winnow.next_pass(permute_bits=True) 
        bob_winnow.next_pass(permute_bits=True) 

        for i in range(alice_winnow._num_of_blocks):
            block_size = alice_winnow._block_size
            startb = i*block_size
            endb = block_size*(i + 1) 
            alice_syndrome = alice_winnow.get_syndrome(i)
            # print(f"Alice syndrome: {alice_winnow.get_syndrome(i)}")
            # print(f"Bob   syndrome: {bob_winnow.get_syndrome(i)}")
            # bob uses Alice's syndrome to fix his key
            # if i % 1000 == 0:
            #     print(f"Bob's key before fix: {bob_key.bits[startb:endb]} block {i}")
            bob_winnow.fix_with_syndrome(i, alice_syndrome)


        qber = get_curr_qber(alice_winnow, bob_winnow)
        print(f"Qber {qber}%")
        qber_list.append(qber)
        num_pass.append(alice_winnow._pass_number)

        return qber_list, num_pass
   








many_block_schedules = [
    [4,4,0,0,0,0,0,0],
    [4,2,2,0,0,0,0,0],
    [2,1,0,1,0,0,0,0],
    [2,2,0,0,0,0,0,0]
]

qber_for_schedule = []
num_pass_for_schedule = []


for block_schedule in many_block_schedules:
    qber_list, num_pass = winnow_pipeline(block_schedule)
    qber_for_schedule.append(qber_list)
    num_pass_for_schedule.append(num_pass)





   

Qber before EC 24.382049560546875%
Qber after first pass 17.305397033691406%
Alice seed: 42
Bob seed:   42
Qber 13.097066243489582%
Qber before EC 24.382049560546875%


In [ ]:


for i in np.range(4):
    plt.plot(num_pass_for_schedule[i], qber_for_schedule[i], label = f"{many_block_schedules[i]}")
    plt.legend()
    plt.show()